# APWC rolling z-score gate付きテーマ・モメンタム戦略

このノートブックは、以下の3つの wide 型データを前提に、APWC のローリング z-score を regime filter として使うテーマ・モメンタム戦略を構築・検証するためのものです。

- `apwc`: テーマバスケットの APWC 時系列。index は日付、columns はテーマ名。
- `total_ret`: テーマバスケットのトータルリターン系列。index は日付、columns はテーマ名。
- `resid_ret`: テーマバスケットの残差リターン系列。index は日付、columns はテーマ名。

狙いは、APWC がそのテーマ自身の過去分布に対して高い局面では、残差リターン・モメンタムが効きやすい、という仮定を確認し、それに基づくロング・ショート戦略を作ることです。

ここで使う `apwc_z` は formal な bootstrap 検定の z-score ではなく、APWC 時系列のローリング標準化です。したがって、このノートブックでは `apwc_z >= threshold` を「統計的有意性」ではなく「coherence regime の proxy」として扱います。

## 0. 基本設計

記法は以下です。

\[
A_{t,i}: \text{theme }i\text{ の APWC}
\]

\[
R_{t,i}: \text{theme }i\text{ の total return}
\]

\[
U_{t,i}: \text{theme }i\text{ の residual return}
\]

APWC のローリング z-score は、

\[
Z_{t,i}=\frac{A_{t,i}-\mu_{t,i}^{(L)}}{\sigma_{t,i}^{(L)}}
\]

で定義します。

active universe は、まず以下で定義します。

\[
G_{t,i}=1[Z_{t,i}\geq z_0]\cdot 1[A_{t,i}>\text{APWC floor}]
\]

方向は residual momentum で決めます。

\[
M_{t,i}^{(K)}=\sum_{s=t-K+1}^{t} U_{s,i}
\]

基本戦略の score は、

\[
S_{t,i}=G_{t,i}\cdot \operatorname{clip}\left(\frac{M_{t,i}^{(K)}}{\hat{\sigma}_{t,i}\sqrt{K}},-3,3\right)
\]

です。weight は score を gross exposure で正規化します。

In [ ]:
from __future__ import annotations

from pathlib import Path
from itertools import product
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:.6f}".format

## 1. パラメータ

最初は論文の検証設定に合わせやすい `60営業日` momentum / forward horizon と、約1年の `252営業日` APWC z-score window を baseline にします。

In [ ]:
# -----------------------------
# User settings
# -----------------------------

# 実データを読む場合は False にして、下のパスを設定してください。
# 初期状態ではノートブックが end-to-end で動くように synthetic sample data を使います。
USE_SAMPLE_DATA = True

# CSV / Excel / Parquet に対応しています。
APWC_PATH = Path("apwc.csv")
TOTAL_RET_PATH = Path("total_ret.csv")
RESID_RET_PATH = Path("resid_ret.csv")

# Excel の場合に使います。
APWC_SHEET = 0
TOTAL_RET_SHEET = 0
RESID_RET_SHEET = 0

# wide データの読み込み時に、日付列を index にするための指定。
# CSV / Excel の先頭列が日付なら 0 のままで問題ありません。
DATE_COL = 0

# Baseline parameters
Z_WINDOW = 252
Z_THRESHOLD = 2.0
APWC_FLOOR = 0.0

MOM_WINDOW = 60
VOL_WINDOW = 60
FORWARD_HORIZON = 60

GROSS_EXPOSURE = 1.0
MAX_ABS_WEIGHT = 0.10
MIN_ACTIVE_THEMES = 3
LONG_ONLY = False
REBALANCE_FREQ = None  # None, "W-FRI", "M" など
TRANSACTION_COST_BPS = 0.0

# パラメータグリッドは重くなりやすいため、必要なときだけ True にしてください。
RUN_PARAMETER_GRID = False

OUTPUT_DIR = Path("outputs")

## 2. データ読み込みユーティリティ

wide 型の DataFrame を読み込み、index を `DatetimeIndex`、columns をテーマ名として扱います。

In [ ]:
def read_wide(path: str | Path,
              date_col: int | str = 0,
              sheet_name: int | str = 0) -> pd.DataFrame:
    """
    wide型データを読み込む。

    Parameters
    ----------
    path:
        .csv, .xlsx, .xls, .parquet のいずれか。
    date_col:
        日付列。通常は 0。
    sheet_name:
        Excel の場合の sheet 指定。

    Returns
    -------
    pd.DataFrame
        index: DatetimeIndex
        columns: theme names
        values: numeric
    """
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path, index_col=date_col, parse_dates=True)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, index_col=date_col, parse_dates=True, sheet_name=sheet_name)
    elif suffix == ".parquet":
        df = pd.read_parquet(path)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.loc[~df.index.duplicated(keep="last")]
    return df


def validate_wide(df: pd.DataFrame, name: str) -> None:
    """wide型データの最低限の検証。"""
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame.")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}.index must be a DatetimeIndex.")
    if df.columns.duplicated().any():
        dupes = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"{name} has duplicated columns: {dupes[:5]}")
    if df.index.duplicated().any():
        raise ValueError(f"{name} has duplicated dates in index.")


def align_wide(apwc: pd.DataFrame,
               total_ret: pd.DataFrame,
               resid_ret: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    3つのwide DataFrameを共通index・共通columnsにそろえる。
    """
    validate_wide(apwc, "apwc")
    validate_wide(total_ret, "total_ret")
    validate_wide(resid_ret, "resid_ret")

    common_idx = apwc.index.intersection(total_ret.index).intersection(resid_ret.index)
    common_cols = apwc.columns.intersection(total_ret.columns).intersection(resid_ret.columns)

    if len(common_idx) == 0:
        raise ValueError("No common dates across apwc, total_ret, and resid_ret.")
    if len(common_cols) == 0:
        raise ValueError("No common theme columns across apwc, total_ret, and resid_ret.")

    apwc = apwc.loc[common_idx, common_cols].sort_index()
    total_ret = total_ret.loc[common_idx, common_cols].sort_index()
    resid_ret = resid_ret.loc[common_idx, common_cols].sort_index()

    return apwc, total_ret, resid_ret

## 3. コア計算関数

In [ ]:
def rolling_z(df: pd.DataFrame,
              window: int,
              min_periods: Optional[int] = None,
              ddof: int = 0) -> pd.DataFrame:
    """各列ごとのローリング z-score。"""
    if min_periods is None:
        min_periods = window

    mu = df.rolling(window, min_periods=min_periods).mean()
    sig = df.rolling(window, min_periods=min_periods).std(ddof=ddof)
    return (df - mu) / sig.replace(0.0, np.nan)


def trailing_sum(ret: pd.DataFrame,
                 window: int,
                 min_periods: Optional[int] = None) -> pd.DataFrame:
    """過去 window 日の単純累積リターン。主に residual return 用。"""
    if min_periods is None:
        min_periods = window
    return ret.rolling(window, min_periods=min_periods).sum()


def forward_sum(ret: pd.DataFrame,
                horizon: int,
                min_periods: Optional[int] = None) -> pd.DataFrame:
    """
    t時点に、t+1からt+horizonまでの将来単純累積リターンを置く。
    主に residual return 用。
    """
    if min_periods is None:
        min_periods = horizon
    return ret.shift(-1).iloc[::-1].rolling(horizon, min_periods=min_periods).sum().iloc[::-1]


def forward_compound(ret: pd.DataFrame,
                     horizon: int,
                     min_periods: Optional[int] = None) -> pd.DataFrame:
    """
    t時点に、t+1からt+horizonまでの将来複利リターンを置く。
    主に total return 用。
    """
    if min_periods is None:
        min_periods = horizon
    safe_ret = ret.where(ret > -1.0)
    log_ret = np.log1p(safe_ret)
    fwd_log = log_ret.shift(-1).iloc[::-1].rolling(horizon, min_periods=min_periods).sum().iloc[::-1]
    return np.expm1(fwd_log)


def slope(y: pd.Series, x: pd.Series) -> float:
    """単回帰 y = alpha + beta x の beta。"""
    xy = pd.concat([y, x], axis=1).dropna()
    if xy.shape[0] < 2:
        return np.nan

    yv = xy.iloc[:, 0].astype(float)
    xv = xy.iloc[:, 1].astype(float)

    x_dm = xv - xv.mean()
    y_dm = yv - yv.mean()
    denom = float(np.dot(x_dm, x_dm))
    if denom == 0.0:
        return np.nan
    return float(np.dot(x_dm, y_dm) / denom)


def winsorize_frame(df: pd.DataFrame,
                    lower: float = -3.0,
                    upper: float = 3.0) -> pd.DataFrame:
    """DataFrame全体の値を上下限でclip。"""
    return df.clip(lower=lower, upper=upper)

## 4. 仮説確認用 panel 作成

`make_research_panel` は、wide 型データを long panel に変換し、以下を同時に持つ DataFrame を作ります。

- `apwc`
- `apwc_z`
- `mom_resid`: 過去 residual return momentum
- `fwd_resid`: 将来 residual return
- `fwd_total`: 将来 total return
- `high_apwc`: APWC rolling z-score gate を通過したか

In [ ]:
def make_research_panel(apwc: pd.DataFrame,
                        total_ret: pd.DataFrame,
                        resid_ret: pd.DataFrame,
                        z_window: int = 252,
                        mom_window: int = 60,
                        horizon: int = 60,
                        z_threshold: float = 2.0,
                        apwc_floor: Optional[float] = 0.0,
                        z_min_periods: Optional[int] = None,
                        mom_min_periods: Optional[int] = None) -> pd.DataFrame:
    """
    仮説確認用のlong panelを作成する。

    apwc_floor=None にすると raw APWC の下限条件を使わない。
    """
    apwc, total_ret, resid_ret = align_wide(apwc, total_ret, resid_ret)

    apwc_z = rolling_z(apwc, z_window, min_periods=z_min_periods)
    mom = trailing_sum(resid_ret, mom_window, min_periods=mom_min_periods)
    fwd_resid = forward_sum(resid_ret, horizon)
    fwd_total = forward_compound(total_ret, horizon)

    panel = pd.concat(
        {
            "apwc": apwc.stack(),
            "apwc_z": apwc_z.stack(),
            "mom_resid": mom.stack(),
            "fwd_resid": fwd_resid.stack(),
            "fwd_total": fwd_total.stack(),
        },
        axis=1,
    ).dropna()

    panel.index.names = ["date", "theme"]

    high_apwc = panel["apwc_z"] >= z_threshold
    if apwc_floor is not None:
        high_apwc = high_apwc & (panel["apwc"] > apwc_floor)

    panel["high_apwc"] = high_apwc
    panel["mom_sign"] = np.sign(panel["mom_resid"])
    panel["signed_fwd_resid"] = panel["mom_sign"] * panel["fwd_resid"]
    panel["signed_fwd_total"] = panel["mom_sign"] * panel["fwd_total"]

    return panel


def summarize_trend(panel: pd.DataFrame) -> pd.DataFrame:
    """
    high APWC群とlow APWC群で、残差モメンタムの将来残差リターン予測力を見る。
    ここでは検定ではなく記述統計のみを返す。
    """
    rows = []

    for flag, g in panel.groupby("high_apwc", dropna=False):
        g = g.dropna(subset=["mom_resid", "fwd_resid", "fwd_total"])
        if g.empty:
            continue

        pos = g[g["mom_resid"] > 0]
        neg = g[g["mom_resid"] < 0]

        beta_resid = slope(g["fwd_resid"], g["mom_resid"])
        beta_total = slope(g["fwd_total"], g["mom_resid"])
        ic_resid = g["mom_resid"].corr(g["fwd_resid"])
        ic_total = g["mom_resid"].corr(g["fwd_total"])

        rows.append({
            "group": "high_apwc" if bool(flag) else "low_apwc",
            "obs": len(g),
            "themes": g.index.get_level_values("theme").nunique(),
            "dates": g.index.get_level_values("date").nunique(),
            "avg_apwc": g["apwc"].mean(),
            "avg_apwc_z": g["apwc_z"].mean(),
            "beta_fwd_resid_on_mom": beta_resid,
            "beta_fwd_total_on_mom": beta_total,
            "resid_IC": ic_resid,
            "total_IC": ic_total,
            "avg_fwd_resid_after_pos_mom": pos["fwd_resid"].mean(),
            "avg_fwd_resid_after_neg_mom": neg["fwd_resid"].mean(),
            "resid_pos_minus_neg_spread": pos["fwd_resid"].mean() - neg["fwd_resid"].mean(),
            "avg_signed_future_resid": g["signed_fwd_resid"].mean(),
            "resid_hit_rate": (g["signed_fwd_resid"] > 0).mean(),
            "avg_signed_future_total": g["signed_fwd_total"].mean(),
            "total_hit_rate": (g["signed_fwd_total"] > 0).mean(),
        })

    return pd.DataFrame(rows).set_index("group").sort_index()


def summarize_by_apwc_z_bucket(panel: pd.DataFrame,
                               n_buckets: int = 5) -> pd.DataFrame:
    """APWC z-score bucketごとの residual momentum 有効性を見る。"""
    g = panel.dropna(subset=["apwc_z", "mom_resid", "fwd_resid", "fwd_total"]).copy()
    g["apwc_z_bucket"] = pd.qcut(g["apwc_z"], q=n_buckets, duplicates="drop")

    out = []
    for bucket, x in g.groupby("apwc_z_bucket", observed=False):
        out.append({
            "apwc_z_bucket": str(bucket),
            "obs": len(x),
            "avg_apwc_z": x["apwc_z"].mean(),
            "beta_fwd_resid_on_mom": slope(x["fwd_resid"], x["mom_resid"]),
            "resid_IC": x["mom_resid"].corr(x["fwd_resid"]),
            "avg_signed_future_resid": x["signed_fwd_resid"].mean(),
            "resid_hit_rate": (x["signed_fwd_resid"] > 0).mean(),
        })

    return pd.DataFrame(out).set_index("apwc_z_bucket")

## 5. 戦略構築関数

基本仕様は以下です。

1. APWC rolling z-score で active theme を選別する。
2. 過去 residual return momentum で方向を決める。
3. residual volatility で momentum を標準化する。
4. score を gross exposure で正規化して weight を作る。
5. PnL では look-ahead を避けるため、`weights.shift(1)` を使う。

In [ ]:
def _normalize_one_row(row: pd.Series,
                       gross: float = 1.0,
                       max_abs_weight: Optional[float] = 0.10,
                       min_active: int = 3) -> pd.Series:
    """1時点のscoreをweightに正規化する。"""
    row = row.fillna(0.0)
    active_n = int((row != 0.0).sum())
    if active_n < min_active:
        return row * 0.0

    denom = row.abs().sum()
    if denom == 0.0:
        return row * 0.0

    w = gross * row / denom

    if max_abs_weight is not None:
        w = w.clip(lower=-max_abs_weight, upper=max_abs_weight)

    return w


def apply_rebalance(weights: pd.DataFrame,
                    rebalance_freq: Optional[str] = None) -> pd.DataFrame:
    """
    weightを指定頻度でリバランスする。
    Noneなら日次リバランス。
    例: 'W-FRI', 'M'
    """
    if rebalance_freq is None:
        return weights

    rebalanced = weights.resample(rebalance_freq).last().reindex(weights.index).ffill()
    return rebalanced.fillna(0.0)


def make_momentum_strategy(apwc: pd.DataFrame,
                           total_ret: pd.DataFrame,
                           resid_ret: pd.DataFrame,
                           z_window: int = 252,
                           mom_window: int = 60,
                           vol_window: int = 60,
                           z_threshold: float = 2.0,
                           apwc_floor: Optional[float] = 0.0,
                           gross: float = 1.0,
                           max_abs_weight: Optional[float] = 0.10,
                           min_active: int = 3,
                           long_only: bool = False,
                           use_apwc_gate: bool = True,
                           use_apwc_intensity: bool = False,
                           winsor_lower: float = -3.0,
                           winsor_upper: float = 3.0,
                           rebalance_freq: Optional[str] = None,
                           transaction_cost_bps: float = 0.0) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    APWC z-score gate付きテーマ残差モメンタム戦略。

    Returns
    -------
    weights:
        日次テーマweight。
    pnl:
        日次PnL DataFrame。columns: total, residual, total_net, residual_net, turnover, cost
    diagnostics:
        apwc_z, mom, resid_vol, mom_score, active, score など。
    """
    apwc, total_ret, resid_ret = align_wide(apwc, total_ret, resid_ret)

    apwc_z = rolling_z(apwc, z_window)

    # 残差モメンタムとリスク調整
    mom = resid_ret.rolling(mom_window, min_periods=mom_window).sum()
    resid_vol = resid_ret.rolling(vol_window, min_periods=vol_window).std(ddof=0)
    mom_score = mom / (resid_vol * np.sqrt(mom_window)).replace(0.0, np.nan)
    mom_score = winsorize_frame(mom_score, winsor_lower, winsor_upper)

    if use_apwc_gate:
        active = apwc_z >= z_threshold
        if apwc_floor is not None:
            active = active & (apwc > apwc_floor)
    else:
        active = pd.DataFrame(True, index=apwc.index, columns=apwc.columns)

    score = mom_score.where(active, 0.0)

    if use_apwc_intensity:
        # thresholdを超えた分だけsignalを強くする。
        # 過剰最適化になりやすいので、baselineではFalse推奨。
        intensity = (apwc_z - z_threshold).clip(lower=0.0)
        if not use_apwc_gate:
            intensity = apwc_z.clip(lower=0.0)
        score = score * intensity

    if long_only:
        score = score.clip(lower=0.0)

    weights = score.apply(
        _normalize_one_row,
        axis=1,
        gross=gross,
        max_abs_weight=max_abs_weight,
        min_active=min_active,
    )

    weights = apply_rebalance(weights, rebalance_freq=rebalance_freq)

    # t時点までの情報で作ったweightを、t+1ではなく「翌営業日の実現リターン」に適用するためshift。
    lagged_weights = weights.shift(1).fillna(0.0)

    pnl_total = (lagged_weights * total_ret).sum(axis=1, min_count=1)
    pnl_resid = (lagged_weights * resid_ret).sum(axis=1, min_count=1)

    # 売買コスト: weight変化額 × bps。
    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * (transaction_cost_bps / 10_000.0)

    pnl = pd.DataFrame({
        "total": pnl_total,
        "residual": pnl_resid,
        "turnover": turnover,
        "cost": cost,
        "total_net": pnl_total - cost,
        "residual_net": pnl_resid - cost,
    })

    diagnostics = {
        "apwc_z": apwc_z,
        "mom": mom,
        "resid_vol": resid_vol,
        "mom_score": mom_score,
        "active": active,
        "score": score,
    }

    return weights, pnl, diagnostics

## 6. パフォーマンス評価関数

In [ ]:
def drawdown_series(ret: pd.Series) -> pd.Series:
    """累積リターンからdrawdown系列を作る。"""
    ret = ret.dropna()
    if ret.empty:
        return pd.Series(dtype=float)
    wealth = (1.0 + ret).cumprod()
    return wealth / wealth.cummax() - 1.0


def performance_stats(ret: pd.Series,
                      ann_factor: int = 252) -> pd.Series:
    """日次リターン系列の主要パフォーマンス指標。"""
    ret = ret.dropna()
    if len(ret) == 0:
        return pd.Series(dtype=float)

    wealth = (1.0 + ret).cumprod()
    dd = wealth / wealth.cummax() - 1.0

    avg = ret.mean()
    vol_daily = ret.std(ddof=0)
    ann_vol = vol_daily * np.sqrt(ann_factor)
    ann_return = wealth.iloc[-1] ** (ann_factor / len(ret)) - 1.0

    downside = ret.where(ret < 0.0, 0.0).std(ddof=0) * np.sqrt(ann_factor)

    return pd.Series({
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": np.nan if vol_daily == 0.0 else avg / vol_daily * np.sqrt(ann_factor),
        "sortino": np.nan if downside == 0.0 else ann_return / downside,
        "max_drawdown": dd.min(),
        "calmar": np.nan if dd.min() == 0.0 else ann_return / abs(dd.min()),
        "daily_hit_rate": (ret > 0.0).mean(),
        "avg_daily_return": avg,
        "daily_vol": vol_daily,
        "skew": ret.skew(),
        "kurtosis": ret.kurtosis(),
        "cumulative_return": wealth.iloc[-1] - 1.0,
        "n_days": len(ret),
    })


def turnover_stats(weights: pd.DataFrame) -> pd.Series:
    """weightのturnoverとexposureを要約する。"""
    turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
    gross = weights.abs().sum(axis=1)
    net = weights.sum(axis=1)
    active_n = (weights.abs() > 0.0).sum(axis=1)

    return pd.Series({
        "avg_daily_turnover": turnover.mean(),
        "median_daily_turnover": turnover.median(),
        "avg_gross_exposure": gross.mean(),
        "avg_abs_net_exposure": net.abs().mean(),
        "avg_active_themes": active_n.mean(),
        "median_active_themes": active_n.median(),
        "max_active_themes": active_n.max(),
    })


def summarize_strategy(weights: pd.DataFrame,
                       pnl: pd.DataFrame,
                       columns: Iterable[str] = ("residual", "total", "residual_net", "total_net")) -> pd.DataFrame:
    """複数PnL列のperformance statsを横断比較する。"""
    rows = {}
    for col in columns:
        if col in pnl.columns:
            rows[col] = performance_stats(pnl[col])
    perf = pd.DataFrame(rows).T
    perf = pd.concat([perf, pd.DataFrame({"turnover_summary": turnover_stats(weights)})], axis=0)
    return perf

## 7. プロット関数

In [ ]:
def plot_cumulative_returns(returns: Dict[str, pd.Series],
                            title: str = "Cumulative returns") -> None:
    """複数リターン系列の累積リターンを1枚のチャートに描く。"""
    fig, ax = plt.subplots(figsize=(11, 5))
    for label, ret in returns.items():
        ret = ret.dropna()
        if ret.empty:
            continue
        wealth = (1.0 + ret).cumprod() - 1.0
        wealth.plot(ax=ax, label=label)
    ax.set_title(title)
    ax.set_ylabel("Cumulative return")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_drawdowns(returns: Dict[str, pd.Series],
                   title: str = "Drawdowns") -> None:
    """複数リターン系列のdrawdownを1枚のチャートに描く。"""
    fig, ax = plt.subplots(figsize=(11, 5))
    for label, ret in returns.items():
        dd = drawdown_series(ret)
        if dd.empty:
            continue
        dd.plot(ax=ax, label=label)
    ax.set_title(title)
    ax.set_ylabel("Drawdown")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_active_theme_count(active: pd.DataFrame,
                            title: str = "Number of active themes") -> None:
    """APWC gateを通過したテーマ数を描く。"""
    fig, ax = plt.subplots(figsize=(11, 4))
    active_count = active.sum(axis=1)
    active_count.plot(ax=ax)
    ax.set_title(title)
    ax.set_ylabel("count")
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_exposures(weights: pd.DataFrame,
                   title: str = "Gross and net exposures") -> None:
    """gross/net exposureを描く。"""
    fig, ax = plt.subplots(figsize=(11, 4))
    weights.abs().sum(axis=1).plot(ax=ax, label="gross")
    weights.sum(axis=1).plot(ax=ax, label="net")
    ax.set_title(title)
    ax.set_ylabel("exposure")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()

## 8. Synthetic sample data

手元データをまだ接続していない状態でもノートブックが実行できるよう、簡単な synthetic data generator を用意します。

実分析では `USE_SAMPLE_DATA = False` にし、`APWC_PATH`, `TOTAL_RET_PATH`, `RESID_RET_PATH` を自分のファイルに差し替えてください。

In [ ]:
def make_sample_data(n_days: int = 1_250,
                     n_themes: int = 30,
                     seed: int = 7) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    ノートブック動作確認用の簡易synthetic data。
    実証結果を示すものではない。
    """
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2019-01-01", periods=n_days)
    themes = [f"Theme_{i:02d}" for i in range(1, n_themes + 1)]

    # baseline returns
    market = rng.normal(0.0002, 0.006, size=(n_days, 1))
    resid = rng.normal(0.0, 0.008, size=(n_days, n_themes))
    total = 0.3 * market + resid + rng.normal(0.0, 0.004, size=(n_days, n_themes))

    # APWC baseline: around zero
    apwc = rng.normal(0.0, 0.008, size=(n_days, n_themes))

    # Add intermittent coherent regimes with mild residual momentum
    for j in range(n_themes):
        n_regimes = rng.integers(1, 4)
        for _ in range(n_regimes):
            start = int(rng.integers(120, max(121, n_days - 180)))
            length = int(rng.integers(50, 140))
            end = min(n_days, start + length)
            direction = rng.choice([-1.0, 1.0])

            apwc[start:end, j] += rng.normal(0.055, 0.012, size=end - start)

            # trend-like residual drift during high APWC periods
            drift = direction * rng.uniform(0.0006, 0.0012)
            resid[start:end, j] += drift
            total[start:end, j] += drift

            # mild continuation pattern
            for t in range(start + 1, end):
                resid[t, j] += 0.15 * resid[t - 1, j]
                total[t, j] += 0.10 * resid[t - 1, j]

    apwc = pd.DataFrame(apwc, index=dates, columns=themes)
    resid_ret = pd.DataFrame(resid, index=dates, columns=themes)
    total_ret = pd.DataFrame(total, index=dates, columns=themes)
    return apwc, total_ret, resid_ret

## 9. データ読み込み

実データを使う場合は、このセルで `apwc`, `total_ret`, `resid_ret` を作成します。

In [ ]:
if USE_SAMPLE_DATA:
    apwc, total_ret, resid_ret = make_sample_data()
else:
    apwc = read_wide(APWC_PATH, date_col=DATE_COL, sheet_name=APWC_SHEET)
    total_ret = read_wide(TOTAL_RET_PATH, date_col=DATE_COL, sheet_name=TOTAL_RET_SHEET)
    resid_ret = read_wide(RESID_RET_PATH, date_col=DATE_COL, sheet_name=RESID_RET_SHEET)

apwc, total_ret, resid_ret = align_wide(apwc, total_ret, resid_ret)

print("apwc shape      :", apwc.shape)
print("total_ret shape :", total_ret.shape)
print("resid_ret shape :", resid_ret.shape)
print("date range      :", apwc.index.min().date(), "to", apwc.index.max().date())
print("n themes        :", apwc.shape[1])

display(apwc.head())

## 10. 仮説確認: high APWC 群で residual momentum が効きやすいか

まずは戦略化する前に、long panel で以下を確認します。

- `high_apwc` 群の `beta_fwd_resid_on_mom` が正か。
- `high_apwc` 群の `resid_IC` が正か。
- `high_apwc` 群の `avg_signed_future_resid` が正か。
- `low_apwc` 群との差があるか。

In [ ]:
panel = make_research_panel(
    apwc=apwc,
    total_ret=total_ret,
    resid_ret=resid_ret,
    z_window=Z_WINDOW,
    mom_window=MOM_WINDOW,
    horizon=FORWARD_HORIZON,
    z_threshold=Z_THRESHOLD,
    apwc_floor=APWC_FLOOR,
)

trend_summary = summarize_trend(panel)
display(trend_summary)

bucket_summary = summarize_by_apwc_z_bucket(panel, n_buckets=5)
display(bucket_summary)

In [ ]:
def forward_sum(ret: pd.DataFrame, horizon: int):
    """
    t時点に、t+1からt+horizonまでの単純累積リターンを置く。
    residual / total の加法分解を見るために使う。
    """
    return ret.shift(-1).iloc[::-1].rolling(
        horizon, min_periods=horizon
    ).sum().iloc[::-1]


def add_total_return_diagnostics(panel: pd.DataFrame,
                                 total_ret: pd.DataFrame,
                                 resid_ret: pd.DataFrame,
                                 horizon: int = 60):
    """
    panelに将来トータルリターン、将来残差リターン、将来common成分を追加する。
    indexは(date, theme)を想定。
    """
    fwd_total_sum = forward_sum(total_ret, horizon).stack().rename("fwd_total_sum")
    fwd_resid_sum = forward_sum(resid_ret, horizon).stack().rename("fwd_resid_sum")

    out = panel.copy()
    out = out.join(fwd_total_sum)
    out = out.join(fwd_resid_sum)

    # 既存のfwd_residがある場合はそれを優先。なければ新しく作ったfwd_resid_sumを使う。
    if "fwd_resid" not in out.columns:
        out["fwd_resid"] = out["fwd_resid_sum"]

    # total = residual + common と見なす
    out["fwd_common_sum"] = out["fwd_total_sum"] - out["fwd_resid"]

    # residual momentumの方向に合わせた将来リターン
    out["mom_sign"] = np.sign(out["mom_resid"])

    out["signed_fwd_total"] = out["mom_sign"] * out["fwd_total_sum"]
    out["signed_fwd_resid"] = out["mom_sign"] * out["fwd_resid"]
    out["signed_fwd_common"] = out["mom_sign"] * out["fwd_common_sum"]

    return out.dropna(subset=[
        "apwc_z",
        "mom_resid",
        "fwd_total_sum",
        "fwd_resid",
        "fwd_common_sum",
        "signed_fwd_total",
        "signed_fwd_resid",
        "signed_fwd_common",
    ])

In [ ]:
def add_apwc_z_quintile(panel: pd.DataFrame,
                        n_buckets: int = 5,
                        datewise: bool = True):
    """
    APWC z-scoreのquintileを追加する。
    datewise=Trueなら各日付のcross-sectionで5分位を作る。
    """
    out = panel.copy()

    def _qcut_safe(s: pd.Series):
        result = pd.Series(np.nan, index=s.index, dtype=float)
        valid = s.dropna()

        if len(valid) < n_buckets:
            return result

        result.loc[valid.index] = pd.qcut(
            valid.rank(method="first"),
            q=n_buckets,
            labels=False
        ) + 1

        return result

    if datewise:
        out["apwc_z_q"] = out.groupby(level="date")["apwc_z"].transform(_qcut_safe)
    else:
        out["apwc_z_q"] = pd.qcut(
            out["apwc_z"].rank(method="first"),
            q=n_buckets,
            labels=False
        ) + 1

    out["apwc_z_q"] = out["apwc_z_q"].astype("Int64")

    return out.dropna(subset=["apwc_z_q"])

In [ ]:
def slope(y: pd.Series, x: pd.Series):
    x = x.astype(float)
    y = y.astype(float)

    x_dm = x - x.mean()
    y_dm = y - y.mean()

    denom = np.dot(x_dm, x_dm)
    if denom == 0:
        return np.nan

    return np.dot(x_dm, y_dm) / denom


def summarize_total_relation_by_apwc_bucket(panel: pd.DataFrame,
                                            bucket_col: str = "apwc_z_q"):
    rows = []

    for q, g in panel.groupby(bucket_col):
        g = g.dropna(subset=[
            "mom_resid",
            "fwd_total_sum",
            "fwd_resid",
            "fwd_common_sum",
            "signed_fwd_total",
            "signed_fwd_resid",
            "signed_fwd_common",
        ])

        if len(g) == 0:
            continue

        pos = g[g["mom_resid"] > 0]
        neg = g[g["mom_resid"] < 0]

        avg_signed_total = g["signed_fwd_total"].mean()
        avg_signed_resid = g["signed_fwd_resid"].mean()
        avg_signed_common = g["signed_fwd_common"].mean()

        rows.append({
            "apwc_z_q": q,
            "obs": len(g),
            "avg_apwc_z": g["apwc_z"].mean(),

            # residual momentumが将来total returnを説明するか
            "beta_fwd_total_on_resid_mom": slope(
                g["fwd_total_sum"],
                g["mom_resid"]
            ),
            "total_IC": g["mom_resid"].corr(g["fwd_total_sum"]),

            # momentum方向に見た将来リターン
            "avg_signed_fwd_total": avg_signed_total,
            "avg_signed_fwd_resid": avg_signed_resid,
            "avg_signed_fwd_common": avg_signed_common,

            # 正・負モメンタムの将来total return差
            "avg_fwd_total_after_pos_mom": pos["fwd_total_sum"].mean(),
            "avg_fwd_total_after_neg_mom": neg["fwd_total_sum"].mean(),
            "total_pos_minus_neg": (
                pos["fwd_total_sum"].mean() - neg["fwd_total_sum"].mean()
            ),

            # hit rate
            "total_hit_rate": (g["signed_fwd_total"] > 0).mean(),
            "resid_hit_rate": (g["signed_fwd_resid"] > 0).mean(),

            # residual alphaがtotalにどれだけ残っているかのラフな比率
            "total_to_resid_signed_ratio": (
                avg_signed_total / avg_signed_resid
                if avg_signed_resid != 0 else np.nan
            ),
        })

    return pd.DataFrame(rows).set_index("apwc_z_q").sort_index()

In [ ]:
panel_total = add_total_return_diagnostics(
    panel=panel,
    total_ret=total_ret,
    resid_ret=resid_ret,
    horizon=60
)

panel_total = add_apwc_z_quintile(
    panel_total,
    n_buckets=5,
    datewise=True
)

total_relation_summary = summarize_total_relation_by_apwc_bucket(panel_total)
display(total_relation_summary)

In [ ]:
def summarize_datewise_ic_by_bucket(panel: pd.DataFrame,
                                    x_col: str = "mom_resid",
                                    y_col: str = "fwd_total_sum",
                                    bucket_col: str = "apwc_z_q",
                                    min_obs: int = 3):
    tmp = panel.reset_index()
    rows = []

    for (date, q), g in tmp.groupby(["date", bucket_col]):
        g = g.dropna(subset=[x_col, y_col])

        if len(g) < min_obs:
            continue

        ic = g[x_col].corr(g[y_col])

        rows.append({
            "date": date,
            "apwc_z_q": q,
            "ic": ic,
            "obs": len(g),
        })

    ic_df = pd.DataFrame(rows)

    summary = ic_df.groupby("apwc_z_q").agg(
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        pct_positive_ic=("ic", lambda x: (x > 0).mean()),
        n_dates=("ic", "count"),
        avg_obs=("obs", "mean"),
    )

    return summary, ic_df


datewise_total_ic_summary, datewise_total_ic = summarize_datewise_ic_by_bucket(
    panel_total,
    x_col="mom_resid",
    y_col="fwd_total_sum",
    bucket_col="apwc_z_q",
    min_obs=3
)

display(datewise_total_ic_summary)

In [ ]:
def row_qcut_safe(row: pd.Series, n_buckets: int = 5):
    result = pd.Series(np.nan, index=row.index, dtype=float)
    valid = row.dropna()

    if len(valid) < n_buckets:
        return result

    result.loc[valid.index] = pd.qcut(
        valid.rank(method="first"),
        q=n_buckets,
        labels=False
    ) + 1

    return result


def apwc_bucket_momentum_pnl(apwc: pd.DataFrame,
                             total_ret: pd.DataFrame,
                             resid_ret: pd.DataFrame,
                             z_window: int = 252,
                             mom_window: int = 60,
                             vol_window: int = 60,
                             n_buckets: int = 5,
                             gross: float = 1.0):
    """
    APWC z-score quintileごとに、residual momentum戦略を作り、
    total PnL / residual PnL / common PnLを比較する。
    """
    apwc, total_ret, resid_ret = align_wide(apwc, total_ret, resid_ret)

    apwc_z = rolling_z(apwc, z_window)

    mom = resid_ret.rolling(mom_window, min_periods=mom_window).sum()
    vol = resid_ret.rolling(vol_window, min_periods=vol_window).std(ddof=0)

    score = mom / (vol * np.sqrt(mom_window)).replace(0.0, np.nan)
    score = score.clip(lower=-3.0, upper=3.0)

    buckets = apwc_z.apply(
        row_qcut_safe,
        axis=1,
        n_buckets=n_buckets
    )

    pnl = {}

    for q in range(1, n_buckets + 1):
        q_score = score.where(buckets == q, 0.0).fillna(0.0)

        denom = q_score.abs().sum(axis=1).replace(0.0, np.nan)
        weights = gross * q_score.div(denom, axis=0).fillna(0.0)

        pnl_total = (weights.shift(1) * total_ret).sum(axis=1, min_count=1)
        pnl_resid = (weights.shift(1) * resid_ret).sum(axis=1, min_count=1)
        pnl_common = pnl_total - pnl_resid

        pnl[f"Q{q}_total"] = pnl_total
        pnl[f"Q{q}_resid"] = pnl_resid
        pnl[f"Q{q}_common"] = pnl_common

    return pd.DataFrame(pnl)

## 11. Baseline strategy

APWC z-score gate 付き residual momentum 戦略を作成します。

In [ ]:
weights, pnl, diag = make_momentum_strategy(
    apwc=apwc,
    total_ret=total_ret,
    resid_ret=resid_ret,
    z_window=Z_WINDOW,
    mom_window=MOM_WINDOW,
    vol_window=VOL_WINDOW,
    z_threshold=Z_THRESHOLD,
    apwc_floor=APWC_FLOOR,
    gross=GROSS_EXPOSURE,
    max_abs_weight=MAX_ABS_WEIGHT,
    min_active=MIN_ACTIVE_THEMES,
    long_only=LONG_ONLY,
    use_apwc_gate=True,
    use_apwc_intensity=False,
    rebalance_freq=REBALANCE_FREQ,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)

print("Performance summary")
display(summarize_strategy(weights, pnl))

print("Latest weights")
display(weights.tail(1).T.sort_values(weights.index[-1], key=lambda s: s.abs(), ascending=False).head(20))

In [ ]:
plot_cumulative_returns(
    {
        "residual": pnl["residual"],
        "total": pnl["total"],
        "residual_net": pnl["residual_net"],
        "total_net": pnl["total_net"],
    },
    title="Baseline APWC-gated residual momentum strategy",
)

plot_drawdowns(
    {
        "residual": pnl["residual"],
        "total": pnl["total"],
        "residual_net": pnl["residual_net"],
        "total_net": pnl["total_net"],
    },
    title="Baseline strategy drawdowns",
)

plot_active_theme_count(diag["active"], title="APWC-gated active theme count")
plot_exposures(weights, title="Baseline strategy exposures")

## 12. APWC gate の付加価値確認

APWC gate を使わない plain residual momentum と比較します。

この比較で見たいのは、APWC gate 付き戦略が、plain momentum に対して以下で改善しているかです。

- residual PnL の Sharpe
- total PnL の Sharpe
- drawdown
- hit rate
- turnover after cost

In [ ]:
# Plain residual momentum: APWC gateなし
weights_plain, pnl_plain, diag_plain = make_momentum_strategy(
    apwc=apwc,
    total_ret=total_ret,
    resid_ret=resid_ret,
    z_window=Z_WINDOW,
    mom_window=MOM_WINDOW,
    vol_window=VOL_WINDOW,
    z_threshold=Z_THRESHOLD,
    apwc_floor=None,
    gross=GROSS_EXPOSURE,
    max_abs_weight=MAX_ABS_WEIGHT,
    min_active=MIN_ACTIVE_THEMES,
    long_only=LONG_ONLY,
    use_apwc_gate=False,
    use_apwc_intensity=False,
    rebalance_freq=REBALANCE_FREQ,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)

comparison = pd.DataFrame({
    "gated_residual": performance_stats(pnl["residual"]),
    "plain_residual": performance_stats(pnl_plain["residual"]),
    "gated_total": performance_stats(pnl["total"]),
    "plain_total": performance_stats(pnl_plain["total"]),
    "gated_residual_net": performance_stats(pnl["residual_net"]),
    "plain_residual_net": performance_stats(pnl_plain["residual_net"]),
    "gated_total_net": performance_stats(pnl["total_net"]),
    "plain_total_net": performance_stats(pnl_plain["total_net"]),
}).T

display(comparison)

plot_cumulative_returns(
    {
        "gated residual": pnl["residual"],
        "plain residual": pnl_plain["residual"],
        "gated total": pnl["total"],
        "plain total": pnl_plain["total"],
    },
    title="APWC gate vs plain residual momentum",
)

## 13. パラメータ・ロバストネス

`z_window`, `z_threshold`, `mom_window` を変えて、結果が特定パラメータだけに依存していないかを確認します。

In [ ]:
def run_parameter_grid(apwc: pd.DataFrame,
                       total_ret: pd.DataFrame,
                       resid_ret: pd.DataFrame,
                       z_windows: Iterable[int] = (126, 252, 504),
                       z_thresholds: Iterable[float] = (1.0, 1.5, 2.0, 2.5),
                       mom_windows: Iterable[int] = (20, 60, 126),
                       vol_window: int = 60,
                       apwc_floor: Optional[float] = 0.0,
                       gross: float = 1.0,
                       max_abs_weight: Optional[float] = 0.10,
                       min_active: int = 3,
                       long_only: bool = False,
                       transaction_cost_bps: float = 0.0) -> pd.DataFrame:
    """パラメータグリッドを回し、主要統計を返す。"""
    rows = []

    for zw, zt, mw in product(z_windows, z_thresholds, mom_windows):
        try:
            w, p, _ = make_momentum_strategy(
                apwc=apwc,
                total_ret=total_ret,
                resid_ret=resid_ret,
                z_window=zw,
                mom_window=mw,
                vol_window=vol_window,
                z_threshold=zt,
                apwc_floor=apwc_floor,
                gross=gross,
                max_abs_weight=max_abs_weight,
                min_active=min_active,
                long_only=long_only,
                use_apwc_gate=True,
                use_apwc_intensity=False,
                transaction_cost_bps=transaction_cost_bps,
            )

            ps_resid = performance_stats(p["residual_net"])
            ps_total = performance_stats(p["total_net"])
            ts = turnover_stats(w)

            rows.append({
                "z_window": zw,
                "z_threshold": zt,
                "mom_window": mw,
                "resid_net_sharpe": ps_resid.get("sharpe", np.nan),
                "resid_net_ann_return": ps_resid.get("ann_return", np.nan),
                "resid_net_max_dd": ps_resid.get("max_drawdown", np.nan),
                "total_net_sharpe": ps_total.get("sharpe", np.nan),
                "total_net_ann_return": ps_total.get("ann_return", np.nan),
                "total_net_max_dd": ps_total.get("max_drawdown", np.nan),
                "avg_active_themes": ts.get("avg_active_themes", np.nan),
                "avg_daily_turnover": ts.get("avg_daily_turnover", np.nan),
            })
        except Exception as exc:
            rows.append({
                "z_window": zw,
                "z_threshold": zt,
                "mom_window": mw,
                "error": str(exc),
            })

    return pd.DataFrame(rows)

In [ ]:
if RUN_PARAMETER_GRID:
    grid = run_parameter_grid(
        apwc=apwc,
        total_ret=total_ret,
        resid_ret=resid_ret,
        z_windows=(126, 252, 504),
        z_thresholds=(1.0, 1.5, 2.0, 2.5),
        mom_windows=(20, 60, 126),
        vol_window=VOL_WINDOW,
        apwc_floor=APWC_FLOOR,
        gross=GROSS_EXPOSURE,
        max_abs_weight=MAX_ABS_WEIGHT,
        min_active=MIN_ACTIVE_THEMES,
        long_only=LONG_ONLY,
        transaction_cost_bps=TRANSACTION_COST_BPS,
    )

    # residual_net_sharpe上位を確認
    cols = [
        "z_window", "z_threshold", "mom_window",
        "resid_net_sharpe", "resid_net_ann_return", "resid_net_max_dd",
        "total_net_sharpe", "total_net_ann_return", "total_net_max_dd",
        "avg_active_themes", "avg_daily_turnover",
    ]
    display(grid[cols].sort_values("resid_net_sharpe", ascending=False).head(20))
else:
    grid = pd.DataFrame()
    print("RUN_PARAMETER_GRID=False. Set it to True to run robustness grid search.")

## 14. 結果保存

必要に応じて、weights、PnL、summary、grid search を CSV に保存します。

In [ ]:
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    weights.to_csv(OUTPUT_DIR / "weights_apwc_gated_momentum.csv")
    pnl.to_csv(OUTPUT_DIR / "pnl_apwc_gated_momentum.csv")
    trend_summary.to_csv(OUTPUT_DIR / "trend_summary_high_low_apwc.csv")
    bucket_summary.to_csv(OUTPUT_DIR / "trend_summary_by_apwc_z_bucket.csv")
    comparison.to_csv(OUTPUT_DIR / "strategy_comparison_gated_vs_plain.csv")
    grid.to_csv(OUTPUT_DIR / "parameter_grid.csv", index=False)
    print(f"Saved outputs to {OUTPUT_DIR.resolve()}")
else:
    print("SAVE_OUTPUTS=False. Set it to True to save CSV outputs.")

## 15. 実務上のチェックリスト

1. **look-ahead bias**: APWC, residual momentum, residual volatility はすべて t 時点までのデータで作り、PnL には `weights.shift(1)` を使う。
2. **APWC z-score の解釈**: これは bootstrap 検定ではなく、rolling regime indicator。
3. **raw APWC floor**: `APWC_FLOOR=0.0` 以上を推奨。負の APWC が過去対比で高くなっただけのケースを除く。
4. **残差PnLとトータルPnLを分ける**: 仮説確認は residual PnL、実装可能性は total PnL。
5. **plain momentum との比較**: APWC gate が本当に付加価値を持つか確認する。
6. **テーマ重複**: テーマ間で構成銘柄が重複している場合、テーマリターン相関や構成銘柄ベースでクラスタ上限を追加する。
7. **取引コスト**: `TRANSACTION_COST_BPS` を現実的な水準にして net performance を見る。
8. **パラメータ依存性**: `z_window`, `z_threshold`, `mom_window` を変えた robustness を見る。